[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Advanced-Time-Series/blob/main/Quantlets/Ch_13/ATS_ch13_conformal_time/ATS_ch13_conformal_time.ipynb)

# ATS_ch13_conformal_time

Conformal prediction for dependent data: weighted conformal under changepoints (Barber et al. 2023); the volatility design of Gibbs and Candes (2021) with ACI on the S&P 500, BET and NVIDIA; the step size gamma; conditional coverage by volatility regime; quantile tracking, conformal PID (Angelopoulos, Candes and Tibshirani 2023) and EnbPI (Xu and Xie 2021) on Romanian day-ahead load.

*Advanced Time Series Analysis and Forecasting (ATS), Chapter 13: Foundation models and conformal prediction. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ATS/Chapter_13'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# open foundation models on a CPU (Colab: installed here if missing; a model that does not install is skipped)
%pip install -q chronos-forecasting timesfm tirex-ts

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)
LightBlue = '#7EA6E0'         # thin background series (e.g. the other countries of a panel): a blue tint, never grey

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import ats_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
# local copy of the Oxford-Man realized library, if present (not distributed; read_omi downloads it)
REALIZED_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..')
                     if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')
END = '2026-09-18'          # last day of the saved data
# Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and Sheppard 2009): the last public
# file (28 February 2022), preserved by the Internet Archive; six indices and the main measures
OMI_URL = ('https://web.archive.org/web/20220301022212id_/'
           'https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip')
OMI_FILE = 'omi_realized_library_v03.csv'
OMI_SYMBOLS = ['.SPX', '.GDAXI', '.FTSE', '.N225', '.STOXX50E', '.FCHI']
OMI_COLS = ['rv5', 'rv5_ss', 'rv10', 'bv', 'bv_ss', 'medrv', 'rk_parzen', 'rk_twoscale', 'rsv', 'open_to_close',
            'close_price', 'open_price', 'nobs']

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the ATS repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_omi(symbols=None, local=True):
    """Oxford-Man Institute's realized library, version 0.3: daily realised measures (OMI_COLS) of six indices
    (OMI_SYMBOLS), columns date, symbol and the measures, sorted by symbol and date. The library is not
    redistributed with the course: a local copy data/realized/omi_realized_library_v03.csv is used if present,
    otherwise the last public file (Internet Archive copy of 28 February 2022) is downloaded once, the six indices
    are extracted and cached in the temporary folder. Cite: Heber, G., Lunde, A., Shephard, N. and Sheppard, K.
    (2009), Oxford-Man Institute's realized library, version 0.3, Oxford-Man Institute, University of Oxford."""
    import tempfile
    import zipfile
    if 'omi' not in _CACHE:
        cache = os.path.join(tempfile.gettempdir(), 'ats_omi')
        cands = ([os.path.join(REALIZED_DIR, OMI_FILE)] if (local and REALIZED_DIR) else []) + [os.path.join(cache, OMI_FILE)]
        path = next((c for c in cands if os.path.exists(c)), '')
        if not path:
            print("Downloading the Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and "
                  "Sheppard 2009), last public file of 28 February 2022, from the Internet Archive")
            os.makedirs(cache, exist_ok=True)
            zpath = os.path.join(cache, 'oxfordmanrealizedvolatilityindices.zip')
            if not os.path.exists(zpath):
                req = urllib.request.Request(OMI_URL, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=300) as r:
                    raw = r.read()
                with open(zpath, 'wb') as f:
                    f.write(raw)
            with zipfile.ZipFile(zpath) as z:
                d = pd.read_csv(z.open(z.namelist()[0]))
            d = d.rename(columns={d.columns[0]: 'date'})
            d['date'] = pd.to_datetime(d['date'].str[:10])
            d = d[d['Symbol'].isin(OMI_SYMBOLS)][['date', 'Symbol'] + OMI_COLS].rename(columns={'Symbol': 'symbol'})
            path = os.path.join(cache, OMI_FILE)
            d.sort_values(['symbol', 'date']).to_csv(path, index=False, float_format='%.6g')
        _CACHE['omi'] = pd.read_csv(path, parse_dates=['date'])
    d = _CACHE['omi']
    if symbols is not None:
        d = d[d['symbol'].isin([symbols] if isinstance(symbols, str) else list(symbols))]
    return d.copy()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import io
import math
import hashlib
import time
import inspect
import types

try:
    import torch
    from torch import nn
    TORCH = True
except ImportError:
    torch = None
    nn = types.SimpleNamespace(Module=object)
    TORCH = False

SEED = 2026

REAL_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/realized/'

REAL_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..') if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')

LOAD_API = 'https://api.energy-charts.info/public_power?country=ro&start={a}&end={b}'

METEO_API = 'https://archive-api.open-meteo.com/v1/archive?latitude=44.43&longitude=26.10&start_date={a}&end_date={b}&hourly=temperature_2m&timezone=Europe%2FBucharest'

LOAD_YEARS = (2022, 2023, 2024, 2025, 2026)

LOAD_END = '2026-09-30'

LOAD_WIN, LOAD_RES = 364, 182

LOAD_CTX = 2048

EU27 = ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']

HICP = ('prc_hicp_minr', 'M.RCH_A.TOTAL.')

RELEASE = {'Chronos-Bolt': '2024-11-25', 'TiRex': '2025-05-26', 'TimesFM 2.5': '2025-09-02', 'Chronos-2': '2025-10-30'}

PRE_END, POST_START = '2024-11-24', '2025-11-01'

FMS = ['Chronos-Bolt small', 'Chronos-2', 'TimesFM 2.5', 'TiRex']

FM_COL = {'Chronos-Bolt small': st.Orange, 'Chronos-2': st.IDAred, 'TimesFM 2.5': st.Purple, 'TiRex': st.Teal, 'Chronos-Bolt tiny': st.Amber, 'Chronos-Bolt mini': st.Amber, 'Chronos-Bolt base': st.Amber}

_MEM = {}

FM_SPECS = {'Chronos-Bolt tiny': ('chronos', 'amazon/chronos-bolt-tiny', 'deciles'), 'Chronos-Bolt mini': ('chronos', 'amazon/chronos-bolt-mini', 'deciles'), 'Chronos-Bolt small': ('chronos', 'amazon/chronos-bolt-small', 'deciles'), 'Chronos-Bolt base': ('chronos', 'amazon/chronos-bolt-base', 'deciles'), 'Chronos-2': ('chronos', 'amazon/chronos-2', 'wide'), 'TimesFM 2.5': ('timesfm', 'google/timesfm-2.5-200m-pytorch', 'deciles'), 'TiRex': ('tirex', 'NX-AI/TiRex', 'deciles')}

DECILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

WIDE = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]

_FM = {}

# shorter windows than the slides (the slides: LOAD_EVAL 2025-01-01, INFL first 2012-01-01, RV_FIRST 2021/2010, RET_FIRST 2016-01-04)

LOAD_EVAL = '2026-03-01'

INFL = dict(start='2000-01-01', first='2022-01-01', H=12)

RV_FIRST = {'btc': '2024-06-03', 'spx': '2020-01-02'}

RET_FIRST = '2023-01-02'

GS_START = '2003-01-01'

def save(name, save_it=True):
    st.check_no_grey(plt.gcf())
    if save_it:
        st.save_fig(name)
    else:
        plt.show()
        plt.close()


def get_bytes(url):
    """Download a public file once per session (no key); optional local cache folder in ATS_CACHE."""
    import hashlib
    import time
    if url in _MEM:
        return _MEM[url]
    cache = os.environ.get('ATS_CACHE')
    path = os.path.join(cache, hashlib.md5(url.encode()).hexdigest()) if cache else None
    if path and os.path.exists(path):
        _MEM[url] = open(path, 'rb').read()
        return _MEM[url]
    for attempt in range(6):                    # the Energy-Charts API limits the request rate
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (ATS course)'})
            _MEM[url] = urllib.request.urlopen(req, timeout=180).read()
            break
        except urllib.error.HTTPError as e:
            if e.code != 429 or attempt == 5:
                raise
            time.sleep(15 * (attempt + 1))
    if path:
        os.makedirs(cache, exist_ok=True)
        open(path, 'wb').write(_MEM[url])
    return _MEM[url]


def fm_cached(key, name, contexts, H, taus, **kw):
    """fm_forecast with an optional disk cache (folder in ATS_FM_CACHE): the forecasts of one experiment and model."""
    cache = os.environ.get('ATS_FM_CACHE')
    path = os.path.join(cache, f'{key}__{name.replace(" ", "_")}.npy') if cache else None
    if path and os.path.exists(path):
        return np.load(path)
    Q = fm_forecast(name, contexts, H, taus, **kw)
    if Q is not None and path:
        os.makedirs(cache, exist_ok=True)
        np.save(path, Q)
    return Q


def read_realized(fname, **kw):
    """A file of data/realized: local copy of the course data, otherwise the ATS repository on GitHub."""
    p = os.path.join(REAL_DIR, fname) if REAL_DIR else ''
    return pd.read_csv(p if p and os.path.exists(p) else REAL_RAW + fname, **kw)


def ro_load_hourly():
    """Romanian hourly electricity load (GW), local time, 2022 to LOAD_END (Energy-Charts, ENTSO-E transparency data):
    15-minute values averaged within each hour; a days x 24 table (DST days: 23 or 25 hours become 24)."""
    if 'load' in _MEM:
        return _MEM['load']
    parts = []
    for y in LOAD_YEARS:
        b = min(f'{y}-12-31', LOAD_END)
        d = json.loads(get_bytes(LOAD_API.format(a=f'{y}-01-01', b=b)))
        load = [x for x in d['production_types'] if x['name'] == 'Load'][0]['data']
        parts.append(pd.Series(load, index=pd.to_datetime(d['unix_seconds'], unit='s', utc=True), dtype=float))
    s = pd.concat(parts).sort_index()
    s = s[~s.index.duplicated()].tz_convert('Europe/Bucharest')
    df = pd.DataFrame({'y': s.values, 'day': s.index.tz_localize(None).normalize(), 'hour': s.index.hour})
    tab = df.groupby(['day', 'hour'])['y'].mean().unstack('hour')
    tab = tab.interpolate(axis=1, limit_direction='both').interpolate(axis=0, limit_direction='both')
    _MEM['load'] = tab.loc[:LOAD_END] / 1000.0
    return _MEM['load']


def bucharest_temperature():
    """Hourly 2 m temperature in Bucharest (Open-Meteo historical archive, ERA5-based), local time, days x 24."""
    if 'temp' in _MEM:
        return _MEM['temp']
    d = json.loads(get_bytes(METEO_API.format(a=f'{LOAD_YEARS[0]}-01-01', b=LOAD_END)))
    t = pd.Series(d['hourly']['temperature_2m'], index=pd.to_datetime(d['hourly']['time']), dtype=float)
    df = pd.DataFrame({'y': t.values, 'day': t.index.normalize(), 'hour': t.index.hour})
    tab = df.groupby(['day', 'hour'])['y'].mean().unstack('hour').interpolate(axis=0, limit_direction='both')
    _MEM['temp'] = tab
    return tab


def orthodox_easter(y):
    """Orthodox Easter (Gregorian date): Meeus' Julian algorithm plus 13 days (valid 1900-2099)."""
    a, b, c = y % 4, y % 7, y % 19
    d = (19 * c + 15) % 30
    e = (2 * a + 4 * b - d + 34) % 7
    mo, da = divmod(d + e + 114, 31)
    return pd.Timestamp(y, mo, da + 1) + pd.Timedelta(days=13)


def ro_holidays(years):
    """Romanian public holidays: fixed dates, Orthodox Good Friday, Easter and Pentecost (Sunday and Monday)."""
    out = []
    for y in years:
        fixed = ['01-01', '01-02', '01-24', '05-01', '06-01', '08-15', '11-30', '12-01', '12-25', '12-26']
        fixed += ['01-06', '01-07'] if y >= 2024 else []
        out += [pd.Timestamp(f'{y}-{m}') for m in fixed]
        e = orthodox_easter(y)
        out += [e + pd.Timedelta(days=k) for k in (-2, 0, 1, 49, 50)]
    return pd.DatetimeIndex(sorted(set(out)))


def eu_hicp():
    """Annual HICP inflation (%), monthly, of the 27 EU countries (Eurostat prc_hicp_minr, all items), common sample."""
    if 'hicp' in _MEM:
        return _MEM['hicp']
    url = ('https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/' + HICP[0] + '/' + HICP[1] + '+'.join(EU27)
           + '?format=SDMX-CSV&startPeriod=1999-01')
    t = pd.read_csv(io.BytesIO(get_bytes(url)))
    t['date'] = pd.to_datetime(t['TIME_PERIOD'] + '-01')
    P = t.pivot_table(index='date', columns='geo', values='OBS_VALUE')[EU27].loc[INFL['start']:]
    _MEM['hicp'] = P.dropna()
    return _MEM['hicp']


def btc_rv():
    """Daily realised variance of Bitcoin from 5-minute Binance returns (data/realized), in %^2; log RV."""
    d = read_realized('binance_daily_realized.csv', parse_dates=['date']).set_index('date')
    rv = d['btc_rv5'].astype(float)
    return rv[rv > 0].dropna()


def spx_rv():
    """Oxford-Man realized library v0.3 (Heber, Lunde, Shephard and Sheppard 2009): daily 5-minute realised variance
    of the S&P 500, in %^2."""
    d = read_omi()
    s = d[d['symbol'] == '.SPX'].set_index('date').sort_index()['rv5'] * 1e4
    return s[s > 0].dropna()


def pinball(y, q, tau):
    """Pinball (quantile) loss of the tau-quantile forecast q."""
    u = np.asarray(y, float) - np.asarray(q, float)
    return np.maximum(tau * u, (tau - 1) * u)


def crps_q(y, Q, taus):
    """CRPS approximated by twice the average pinball loss over the quantile levels taus; Q has the levels in its last
    axis, y broadcasts against Q[..., 0]."""
    y = np.asarray(y, float)[..., None]
    taus = np.asarray(taus, float)
    return 2 * pinball(y, Q, taus).mean(axis=-1)


def wql(y, Q, taus):
    """Weighted quantile loss of Chronos (Ansari et al. 2024): summed pinball losses scaled by the summed |y|."""
    return float(crps_q(y, Q, taus).sum() / np.abs(np.asarray(y, float)).sum())


def interval_score(y, lo, hi, alpha):
    """Interval score of a central (1 - alpha) interval (Winkler 1972; Gneiting and Raftery 2007): width plus 2/alpha
    times the distance by which y falls outside."""
    y, lo, hi = (np.asarray(v, float) for v in (y, lo, hi))
    return (hi - lo) + 2 / alpha * (lo - y) * (y < lo) + 2 / alpha * (y - hi) * (y > hi)


def qlike(rv, f):
    """QLIKE loss of a variance forecast f for the realised variance rv (Patton 2011)."""
    x = np.asarray(rv, float) / np.asarray(f, float)
    return x - np.log(x) - 1


def gmean(x):
    """Geometric mean (the aggregate of relative scores across series; Fleming and Wallace 1986)."""
    x = np.asarray(x, float)
    return float(np.exp(np.mean(np.log(x))))


def hac_var(x, lags, kernel='bartlett'):
    """Long-run variance: Bartlett (Newey-West) or rectangular kernel."""
    x = np.asarray(x, float) - np.mean(x)
    T = len(x)
    v = x @ x / T
    for k in range(1, lags + 1):
        w = 1 - k / (lags + 1) if kernel == 'bartlett' else 1.0
        v += 2 * w * (x[k:] @ x[:-k]) / T
    return v


def dm_test(d, h=1):
    """Diebold-Mariano test of E[d] = 0 for the loss differential d, HAC variance with h - 1 lags (rectangular;
    Bartlett if negative) and the Harvey-Leybourne-Newbold correction with t(T - 1). Two-sided p-value."""
    d = np.asarray(d, float)
    d = d[~np.isnan(d)]
    T = len(d)
    v = hac_var(d, h - 1, 'rect')
    if v <= 0:
        v = hac_var(d, max(h - 1, 1), 'bartlett')
    dm = d.mean() / np.sqrt(v / T)
    hln = np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T) * dm
    return {'T': T, 'dbar': float(d.mean()), 'hln': float(hln), 'p': float(2 * stats.t.sf(abs(hln), T - 1))}


def mcs(losses, alpha=0.10, B=2000, block=7, seed=2026):
    """Model Confidence Set (Hansen, Lunde and Nason 2011), T_max statistic, moving-block bootstrap.
    losses: T x m DataFrame; returns the MCS p-value of every model and the set at level alpha."""
    L = np.asarray(losses, float)
    L = L[~np.isnan(L).any(axis=1)]
    T, m = L.shape
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(T / block))
    idx = np.concatenate([np.arange(s, s + block) for s in rng.integers(0, T - block + 1, size=(B, nb)).ravel()])
    idx = idx.reshape(B, nb * block)[:, :T]
    Lb = L[idx].mean(axis=1)
    alive, pvals, p_run = list(range(m)), {}, 0.0
    while len(alive) > 1:
        Lm = L[:, alive].mean(axis=0)
        dbar = Lm - Lm.mean()
        db = Lb[:, alive] - Lb[:, alive].mean(axis=1, keepdims=True)
        se = np.sqrt(((db - dbar) ** 2).mean(axis=0))
        t = dbar / se
        p = float((((db - dbar) / se).max(axis=1) > t.max()).mean())
        p_run = max(p_run, p)
        worst = alive[int(np.argmax(t))]
        pvals[worst] = p_run
        alive.remove(worst)
    pvals[alive[0]] = 1.0
    cols = list(losses.columns) if hasattr(losses, 'columns') else list(range(m))
    out = {cols[i]: pvals[i] for i in range(m)}
    return {'p': out, 'set': [c for c in cols if out[c] >= alpha]}


def holm(p):
    """Holm (1979) step-down adjusted p-values (family-wise error rate)."""
    p = np.asarray(p, float)
    m = len(p)
    o = np.argsort(p)
    adj = np.empty(m)
    run = 0.0
    for k, i in enumerate(o):
        run = max(run, (m - k) * p[i])
        adj[i] = min(1.0, run)
    return adj


def bh(p):
    """Benjamini-Hochberg (1995) adjusted p-values (false discovery rate)."""
    p = np.asarray(p, float)
    m = len(p)
    o = np.argsort(p)[::-1]
    adj = np.empty(m)
    run = 1.0
    for k, i in enumerate(o):
        run = min(run, p[i] * m / (m - k))
        adj[i] = run
    return adj


def kupiec(hits, a):
    """Unconditional coverage LR test (Kupiec 1995), chi2(1): (LR, p-value)."""
    h = np.asarray(hits, float)
    n, x = len(h), h.sum()
    pi = x / n
    ll0 = x * np.log(a) + (n - x) * np.log(1 - a)
    ll1 = (x * np.log(pi) if x > 0 else 0) + ((n - x) * np.log(1 - pi) if x < n else 0)
    lr = -2 * (ll0 - ll1)
    return float(lr), float(stats.chi2.sf(lr, 1))


def christoffersen(hits, a):
    """Independence and conditional coverage LR tests (Christoffersen 1998): chi2(1) and chi2(2)."""
    h = np.asarray(hits, int)
    h0, h1 = h[:-1], h[1:]
    n00, n01 = np.sum((h0 == 0) & (h1 == 0)), np.sum((h0 == 0) & (h1 == 1))
    n10, n11 = np.sum((h0 == 1) & (h1 == 0)), np.sum((h0 == 1) & (h1 == 1))
    p01 = n01 / max(n00 + n01, 1)
    p11 = n11 / max(n10 + n11, 1)
    p = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)
    xl = lambda k, q: k * np.log(q) if k > 0 else 0.0   # noqa: E731
    l0 = xl(n00 + n10, 1 - p) + xl(n01 + n11, p)
    l1 = xl(n00, 1 - p01) + xl(n01, p01) + xl(n10, 1 - p11) + xl(n11, p11)
    lr_ind = -2 * (l0 - l1)
    lr_cc = kupiec(h[1:], a)[0] + lr_ind
    return {'p_ind': float(stats.chi2.sf(lr_ind, 1)), 'p_cc': float(stats.chi2.sf(lr_cc, 2))}


def seasonal_naive(ctx, H, m):
    """Seasonal naive point forecast: repeat the last season of length m over H steps."""
    ctx = np.asarray(ctx, float)
    return np.array([ctx[len(ctx) - m + (h % m)] for h in range(H)])


def ar_fit(y, p):
    """OLS fit of an AR(p) with intercept: coefficients (c, phi_1..phi_p), residual variance, AIC."""
    y = np.asarray(y, float)
    X = np.column_stack([np.ones(len(y) - p)] + [y[p - j:len(y) - j] for j in range(1, p + 1)])
    Y = y[p:]
    b = np.linalg.lstsq(X, Y, rcond=None)[0]
    e = Y - X @ b
    s2 = e @ e / len(Y)
    return b, s2, len(Y) * np.log(s2) + 2 * (p + 1)


def ar_forecast(y, H, taus, pmax=12, p=None):
    """Iterated AR(p) forecasts (p by AIC on a common sample if not given) with Gaussian quantiles from the
    psi-weight forecast variance; returns (mean (H,), quantiles (H, k), p)."""
    y = np.asarray(y, float)
    if p is None:
        best = None
        for q in range(1, pmax + 1):
            aic = ar_fit(y[pmax - q:], q)[2]
            if best is None or aic < best[0]:
                best = (aic, q)
        p = best[1]
    b, s2, _ = ar_fit(y, p)
    c, phi = b[0], b[1:]
    hist = list(y[-p:])
    mean = []
    for _ in range(H):
        f = c + sum(phi[j] * hist[-1 - j] for j in range(p))
        mean.append(f)
        hist.append(f)
    psi = [1.0]
    for h in range(1, H):
        psi.append(sum(phi[j] * psi[h - 1 - j] for j in range(min(p, h))))
    sd = np.sqrt(s2 * np.cumsum(np.square(psi)))
    mean = np.array(mean)
    return mean, mean[:, None] + sd[:, None] * stats.norm.ppf(np.asarray(taus))[None, :], p


def ets_forecast(y, H, taus, seasonal_periods=None, reps=400, seed=0):
    """Damped additive-trend ETS (statsmodels ETSModel, maximum likelihood) with simulated predictive quantiles;
    additive seasonality if seasonal_periods is given. Returns (mean (H,), quantiles (H, k))."""
    from statsmodels.tsa.exponential_smoothing.ets import ETSModel
    import warnings
    y = np.asarray(y, float)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        mod = ETSModel(y, error='add', trend='add', damped_trend=True,
                       seasonal='add' if seasonal_periods else None, seasonal_periods=seasonal_periods)
        res = mod.fit(disp=False, maxiter=200)
        sims = res.simulate(H, repetitions=reps, anchor='end', random_state=seed)
    sims = np.asarray(sims).reshape(H, -1)
    return np.asarray(res.forecast(H)), np.quantile(sims, taus, axis=1).T


def har_design(x):
    """HAR regressors (Corsi 2009) of a daily series x (log realised variance): x_t, mean of the last 5 and 22 days."""
    x = np.asarray(x, float)
    d = x[21:]
    w = np.array([x[t - 4:t + 1].mean() for t in range(21, len(x))])
    m = np.array([x[t - 21:t + 1].mean() for t in range(21, len(x))])
    return np.column_stack([np.ones(len(d)), d, w, m])


def har_forecast(x, h=1):
    """HAR forecast of x at t + h (direct regression of x_{t+h} on the HAR regressors), with the residual s.d."""
    x = np.asarray(x, float)
    X = har_design(x)
    Y = x[21 + h:]
    b = np.linalg.lstsq(X[:-h], Y, rcond=None)[0]
    e = Y - X[:-h] @ b
    return float(X[-1] @ b), float(e.std(ddof=4))


def garch11_fit(r, dist='t'):
    """QML fit of a GARCH(1,1) with constant mean zero (returns in %): Student-t (dist='t') or Normal innovations.
    Returns (omega, alpha, beta, nu) with nu = inf for the Normal."""
    r = np.asarray(r, float)
    v0 = r.var()

    def nll(p):
        om, a, b = np.exp(p[0]), 1 / (1 + np.exp(-p[1])), 1 / (1 + np.exp(-p[2]))
        if a + b >= 0.9999:
            return 1e10
        nu = 2.05 + np.exp(p[3]) if dist == 't' else np.inf
        h = np.empty(len(r))
        h[0] = v0
        for t in range(1, len(r)):
            h[t] = om + a * r[t - 1] ** 2 + b * h[t - 1]
        if dist == 't':
            ll = (math.lgamma((nu + 1) / 2) - math.lgamma(nu / 2) - 0.5 * np.log(np.pi * (nu - 2) * h)
                  - (nu + 1) / 2 * np.log1p(r ** 2 / ((nu - 2) * h)))
            return -ll.sum()
        return 0.5 * np.sum(np.log(2 * np.pi * h) + r ** 2 / h)
    x0 = [np.log(0.05 * v0), np.log(0.08 / 0.92), np.log(0.9 / 0.1)] + ([np.log(6.0)] if dist == 't' else [])
    res = optimize.minimize(nll, x0, method='Nelder-Mead',
                            options={'maxiter': 3000, 'xatol': 1e-6, 'fatol': 1e-6})
    p = res.x
    nu = 2.05 + np.exp(p[3]) if dist == 't' else np.inf
    return float(np.exp(p[0])), float(1 / (1 + np.exp(-p[1]))), float(1 / (1 + np.exp(-p[2]))), float(nu)


def garch11_filter(r, par):
    """Conditional variances h_1..h_{T+1} of a GARCH(1,1) for returns r (the last one is the one-step forecast)."""
    om, a, b, _ = par
    r = np.asarray(r, float)
    h = np.empty(len(r) + 1)
    h[0] = r.var()
    for t in range(1, len(r) + 1):
        h[t] = om + a * r[t - 1] ** 2 + b * h[t - 1]
    return h


def set_seed(seed):
    np.random.seed(seed)
    if TORCH:
        torch.manual_seed(seed)


class DLinear(nn.Module):
    """DLinear (Zeng et al. 2023): moving-average trend plus remainder, one linear map from L inputs to H outputs each."""

    def __init__(self, L, H, k=25):
        super().__init__()
        self.k = k
        self.lin_t = nn.Linear(L, H)
        self.lin_s = nn.Linear(L, H)

    def forward(self, x):
        pad = (self.k - 1) // 2
        xp = torch.cat([x[:, :1].repeat(1, pad), x, x[:, -1:].repeat(1, self.k - 1 - pad)], dim=1)
        trend = xp.unfold(1, self.k, 1).mean(-1)
        return self.lin_t(trend) + self.lin_s(x - trend)


class NBeats(nn.Module):
    """A small generic N-BEATS (Oreshkin et al. 2020): stacked fully connected blocks with backcast and forecast."""

    def __init__(self, L, H, width=128, blocks=3, layers=3):
        super().__init__()
        self.blocks = nn.ModuleList()
        for _ in range(blocks):
            mods, d = [], L
            for _ in range(layers):
                mods += [nn.Linear(d, width), nn.ReLU()]
                d = width
            self.blocks.append(nn.ModuleDict({'fc': nn.Sequential(*mods), 'b': nn.Linear(width, L), 'f': nn.Linear(width, H)}))

    def forward(self, x):
        res, out = x, 0
        for bl in self.blocks:
            h = bl['fc'](res)
            res = res - bl['b'](h)
            out = out + bl['f'](h)
        return out


def fit_global(model, X, Y, epochs=40, lr=1e-3, batch=256, seed=0):
    """Train a global network on windows (X: n x L, Y: n x H, already scaled) with Adam and the MAE loss;
    the last 10% of the windows (in time) is the validation set for early stopping."""
    set_seed(seed)
    X, Y = torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)
    nv = max(1, int(0.1 * len(X)))
    Xt, Yt, Xv, Yv = X[:-nv], Y[:-nv], X[-nv:], Y[-nv:]
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best, state, wait = np.inf, None, 0
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(Xt), batch):
            j = perm[i:i + batch]
            opt.zero_grad()
            loss = (model(Xt[j]) - Yt[j]).abs().mean()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            v = float((model(Xv) - Yv).abs().mean())
        if v < best - 1e-6:
            best, state, wait = v, {k: t.clone() for k, t in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 5:
                break
    model.load_state_dict(state)
    model.eval()
    return model


def predict_global(model, X):
    with torch.no_grad():
        return model(torch.tensor(np.asarray(X), dtype=torch.float32)).numpy()


def fm_load(name):
    """Load a foundation model once (CPU); None if its library or checkpoint is not available."""
    if name in _FM:
        return _FM[name]
    lib, ckpt, _ = FM_SPECS[name]
    model = None
    try:
        if TORCH:
            torch.set_num_threads(1)
        if lib == 'chronos':
            from chronos import BaseChronosPipeline
            model = BaseChronosPipeline.from_pretrained(ckpt, device_map='cpu', torch_dtype=torch.float32)
        elif lib == 'timesfm':
            import timesfm
            model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(ckpt)
            model.compile(timesfm.ForecastConfig(max_context=2048, max_horizon=128, normalize_inputs=True,
                                                 use_continuous_quantile_head=True, fix_quantile_crossing=True))
        elif lib == 'tirex':
            from tirex import load_model
            model = load_model(ckpt, device='cpu')
    except Exception as e:                       # library missing, no network, incompatible version: skip
        print(f'{name}: not available ({type(e).__name__}); skipped')
        model = None
    _FM[name] = model
    return model


def fm_levels(name):
    """Quantile levels a model can return: 0.1-0.9 (deciles) or 0.01-0.99 (Chronos-2)."""
    return WIDE if FM_SPECS[name][2] == 'wide' else DECILES


def fm_params(name):
    """Number of parameters of a loaded model (millions)."""
    m = fm_load(name)
    if m is None:
        return np.nan
    net = getattr(m, 'model', m)
    if hasattr(net, 'parameters'):
        return sum(p.numel() for p in net.parameters()) / 1e6
    return np.nan


def _interp_levels(Q, have, want):
    """Quantiles at the levels `want` by linear interpolation of the quantile function between the levels `have`
    (NaN outside the range the model was trained on)."""
    have = np.asarray(have, float)
    out = np.full(Q.shape[:-1] + (len(want),), np.nan)
    for k, t in enumerate(want):
        if t < have[0] - 1e-9 or t > have[-1] + 1e-9:
            continue
        j = int(np.clip(np.searchsorted(have, t) - 1, 0, len(have) - 2))
        w = (t - have[j]) / (have[j + 1] - have[j])
        out[..., k] = (1 - w) * Q[..., j] + w * Q[..., j + 1]
    return out


def fm_forecast(name, contexts, H, taus, batch=64, covariates=None, cross_learning=False):
    """Zero-shot quantile forecasts of a foundation model.
    contexts: list of 1-d arrays (histories, possibly of different lengths); H: horizon; taus: quantile levels.
    covariates (Chronos-2 only): list of dicts {'past': {name: array}, 'future': {name: array}} aligned with contexts.
    Returns an array (n, H, len(taus)), or None if the model is not available."""
    m = fm_load(name)
    if m is None:
        return None
    lib = FM_SPECS[name][0]
    have = fm_levels(name)
    out = []
    for i in range(0, len(contexts), batch):
        ctx = [np.asarray(c, np.float32) for c in contexts[i:i + batch]]
        with torch.no_grad():
            if lib == 'chronos' and name == 'Chronos-2':
                if covariates is not None:
                    inp = [{'target': c, 'past_covariates': {k: np.asarray(v, np.float32) for k, v in cv['past'].items()},
                            'future_covariates': {k: np.asarray(v, np.float32) for k, v in cv['future'].items()}}
                           for c, cv in zip(ctx, covariates[i:i + batch])]
                else:
                    inp = [torch.tensor(c) for c in ctx]
                q, _ = m.predict_quantiles(inp, prediction_length=H, quantile_levels=have, cross_learning=cross_learning)
                q = np.stack([np.asarray(x)[0] for x in q])                 # (n, H, k): first (only) target variate
            elif lib == 'chronos':
                q, _ = m.predict_quantiles([torch.tensor(c) for c in ctx], prediction_length=H, quantile_levels=have)
                q = np.asarray(q)
            elif lib == 'timesfm':
                _, qf = m.forecast(horizon=H, inputs=ctx)
                q = np.asarray(qf)[:, :, 1:]                                 # column 0 is the mean
            else:
                q = m.forecast(context=[torch.tensor(c) for c in ctx], prediction_length=H, output_type='numpy')[0]
                q = np.asarray(q)
        out.append(np.sort(q, axis=-1))
    return _interp_levels(np.concatenate(out), have, list(taus))


def chronos_tokenise(x, n_bins=4093, low=-15.0, high=15.0):
    """The Chronos tokeniser (Ansari et al. 2024): mean scaling s = mean|x| over the context, then uniform bins on
    [low, high]; returns (token ids 0..n_bins-1, the dequantised values s * centre, the scale s)."""
    x = np.asarray(x, float)
    s = np.mean(np.abs(x))
    s = s if s > 0 else 1.0
    centres = np.linspace(low, high, n_bins)
    edges = (centres[1:] + centres[:-1]) / 2
    ids = np.searchsorted(edges, x / s, side='right')
    return ids, s * centres[ids], s


def conformal_quantile(scores, alpha, weights=None):
    """The conformal threshold: the (1 - alpha) quantile of the scores with an extra point mass at +infinity
    (weight 1 for split conformal; weights w_i and w_{n+1} = 1 for the weighted version of Barber et al. 2023).
    Unweighted: the ceil((n + 1)(1 - alpha))-th smallest score, +inf if that rank exceeds n."""
    s = np.asarray(scores, float)
    n = len(s)
    if weights is None:
        k = int(np.ceil((n + 1) * (1 - alpha)))
        return np.inf if k > n else float(np.sort(s)[k - 1])
    w = np.asarray(weights, float)
    p = np.append(w, 1.0) / (w.sum() + 1.0)
    o = np.argsort(s)
    c = np.cumsum(p[:-1][o])
    k = np.searchsorted(c, 1 - alpha - 1e-12)
    return np.inf if k >= n else float(s[o][k])


def split_conformal(y_cal, f_cal, f_test, alpha):
    """Split conformal interval with absolute residual scores: f_test +/- the conformal quantile of |y - f| on the
    calibration set (Papadopoulos et al. 2002; Lei et al. 2018)."""
    q = conformal_quantile(np.abs(np.asarray(y_cal) - np.asarray(f_cal)), alpha)
    return np.asarray(f_test) - q, np.asarray(f_test) + q


def cqr(y_cal, lo_cal, hi_cal, lo_test, hi_test, alpha):
    """Conformalized quantile regression (Romano, Patterson and Candes 2019): score max(lo - y, y - hi) on the
    calibration set; the interval [lo - q, hi + q]."""
    s = np.maximum(np.asarray(lo_cal) - y_cal, y_cal - np.asarray(hi_cal))
    q = conformal_quantile(s, alpha)
    return np.asarray(lo_test) - q, np.asarray(hi_test) + q


def online_threshold(scores, alpha, method='static', n_cal=250, window=None, rho=0.99, gamma=0.005, eta=None,
                     KI=None, Csat=None, scorecaster=None, start=None):
    """Online conformal thresholds q_t, each computed from the scores s_1..s_{t-1} only; the set at time t is
    {y : s_t(y) <= q_t}, so the miss indicator is err_t = 1{s_t > q_t}.
    method: 'static'   split conformal on the first n_cal scores, never updated;
            'rolling'  conformal quantile of the last `window` scores (all past scores if window is None);
            'weighted' weights rho^(t - i) on past scores (non-exchangeable conformal, Barber et al. 2023);
            'aci'      adaptive conformal inference (Gibbs and Candes 2021): level alpha_t,
                       alpha_{t+1} = alpha_t + gamma (alpha - err_t), threshold = quantile of the last `window` scores;
            'qt'       quantile tracking (Angelopoulos, Candes and Tibshirani 2023): q_{t+1} = q_t + eta (err_t - alpha);
            'pid'      quantile tracking + integrator r_t(sum(err - alpha)) = KI tan(x log t / (t Csat)) + a scorecaster
                       (a function past_scores -> forecast of the next score), the conformal PID controller.
    Returns (q, alpha_t) arrays of the same length as scores (NaN before `start`)."""
    s = np.asarray(scores, float)
    T = len(s)
    start = n_cal if start is None else start
    q = np.full(T, np.nan)
    at = np.full(T, np.nan)
    a_t = alpha
    if method in ('qt', 'pid'):
        B = np.nanmax(np.abs(s[:start]))
        eta = 0.01 * B if eta is None else eta
        q_t = conformal_quantile(s[:start], alpha)
        KI = B if KI is None else KI
        Csat = 1.0 if Csat is None else Csat
        integ = 0.0
    for t in range(start, T):
        past = s[:t]
        if method == 'static':
            q[t] = conformal_quantile(s[:n_cal], alpha)
        elif method == 'rolling':
            q[t] = conformal_quantile(past if window is None else past[-window:], alpha)
        elif method == 'weighted':
            w = rho ** np.arange(t, 0, -1)
            q[t] = conformal_quantile(past, alpha, weights=w)
        elif method == 'aci':
            at[t] = a_t
            if a_t <= 0:
                q[t] = np.inf
            elif a_t >= 1:
                q[t] = -np.inf
            else:
                q[t] = conformal_quantile(past if window is None else past[-window:], a_t)
            err = float(s[t] > q[t])
            a_t = a_t + gamma * (alpha - err)
        else:
            base = q_t
            if method == 'pid':
                x = integ * np.log(max(t - start + 1, 2)) / ((t - start + 1) * Csat)
                r = KI * np.tan(x) if abs(x) < np.pi / 2 else np.sign(x) * np.inf
                sc = scorecaster(past) if scorecaster is not None else 0.0
                base = q_t + r + sc
            q[t] = base
            err = float(s[t] > q[t])
            q_t = q_t + eta * (err - alpha)
            integ += err - alpha
    return q, at


def ar_scorecaster(p=1, window=500):
    """A scorecaster for conformal PID: the AR(p) forecast of the next score minus the mean score (so that it adds
    the predictable part only), fitted on the last `window` scores."""
    def f(past):
        x = np.asarray(past[-window:], float)
        if len(x) < 5 * (p + 1):
            return 0.0
        b = ar_fit(x, p)[0]
        return float(b[0] + sum(b[1 + j] * x[-1 - j] for j in range(p)) - x.mean())
    return f


def enbpi(X, y, n_train, alpha, B=25, block=24, s=1, seed=0, lam=1.0):
    """EnbPI (Xu and Xie 2021) with ridge regressions on block-bootstrap samples of the first n_train points:
    leave-one-out residuals of the training points form the initial window; at each later t the interval is the
    ensemble mean +/- the (1 - alpha) quantile of the last n_train absolute residuals; the window is updated with
    the new residual every s steps. Returns (centre, half-width) for t >= n_train."""
    rng = np.random.default_rng(seed)
    X, y = np.asarray(X, float), np.asarray(y, float)
    n, d = X.shape
    mu, sd = X[:n_train].mean(0), X[:n_train].std(0) + 1e-12
    Z = np.column_stack([np.ones(n), (X - mu) / sd])
    nb = int(np.ceil(n_train / block))
    preds = np.empty((B, n))
    inbag = np.zeros((B, n_train), bool)
    for b in range(B):
        starts = rng.integers(0, n_train - block + 1, nb)
        idx = np.concatenate([np.arange(a, a + block) for a in starts])[:n_train]
        inbag[b, np.unique(idx)] = True
        Zb, yb = Z[idx], y[idx]
        beta = np.linalg.solve(Zb.T @ Zb + lam * np.eye(d + 1), Zb.T @ yb)
        preds[b] = Z @ beta
    loo = np.array([preds[~inbag[:, i], i].mean() if (~inbag[:, i]).any() else preds[:, i].mean() for i in range(n_train)])
    resid = list(np.abs(y[:n_train] - loo))
    centre = preds[:, n_train:].mean(axis=0)
    half = np.empty(n - n_train)
    new = []
    for k, t in enumerate(range(n_train, n)):
        half[k] = np.quantile(resid[-n_train:], 1 - alpha)
        new.append(abs(y[t] - centre[k]))
        if len(new) == s:
            resid += new
            new = []
    return centre, half


def fig_weighted(save_it=True, n=2000, burn=200, rho=0.99, alpha=0.1, reps=200, seed=SEED):
    """Weighted (non-exchangeable) conformal prediction against standard conformal under changepoints, in the style
    of the simulations of Barber et al. (2023): Y = X'beta_t + N(0, 1), X ~ N(0, I_4), beta changes at t = 500 and
    1500; least squares refitted on all past data; prequential absolute residuals as scores; weights rho^(t - i).
    Coverage of each time point averaged over `reps` replications."""
    rng = np.random.default_rng(seed)
    b1, b2, b3 = np.array([2, 1, 0, 0.]), np.array([0, -2, -1, 0.]), np.array([0, 0, 2, 1.])
    covs = {'standard': np.zeros(n), 'weighted': np.zeros(n)}
    width = {'standard': np.zeros(n), 'weighted': np.zeros(n)}
    for _ in range(reps):
        X = rng.standard_normal((n, 4))
        B = np.where(np.arange(n)[:, None] < 500, b1, np.where(np.arange(n)[:, None] < 1500, b2, b3))
        Y = (X * B).sum(1) + rng.standard_normal(n)
        S = np.full(n, np.nan)
        XtX, Xty = np.eye(4) * 1e-6, np.zeros(4)
        for t in range(n):
            if t >= 10:
                S[t] = abs(Y[t] - X[t] @ np.linalg.solve(XtX, Xty))
            XtX += np.outer(X[t], X[t])
            Xty += X[t] * Y[t]
        for meth in covs:
            q, _ = online_threshold(S[10:], alpha, method='rolling' if meth == 'standard' else 'weighted', rho=rho,
                                    start=burn - 10)
            q = np.r_[np.full(10, np.nan), q]
            covs[meth] += (S <= q) / reps
            width[meth] += np.where(np.isfinite(q), 2 * q, np.nan) / reps
    fig, ax = plt.subplots(figsize=(11, 3.8))
    t = np.arange(n)
    for meth, col in (('standard', st.MainBlue), ('weighted', st.IDAred)):
        ax.plot(t[burn:], pd.Series(covs[meth][burn:]).rolling(20).mean(), color=col, lw=1.3,
                label=f'{meth} conformal' + (f' (rho = {rho})' if meth == 'weighted' else ''))
    ax.axhline(1 - alpha, color=st.Forest, ls='--', lw=1)
    for c in (500, 1500):
        ax.axvline(c, color=st.Amber, ls=':', lw=1.2)
    ax.set_xlabel('time')
    ax.set_ylabel(f'coverage (mean of {reps} runs)')
    st.legend_outside_bottom(ax, ncol=2, y=-0.16)
    plt.tight_layout()
    save('ats_ch13_weighted', save_it)
    after = slice(1500, 1600)
    return {m: dict(cov=float(np.mean(covs[m][burn:])), after=float(np.mean(covs[m][after])),
                    w=float(np.nanmean(width[m][burn:])), min=float(pd.Series(covs[m][burn:]).rolling(20).mean().min()))
            for m in covs}


def garch_scores(name, start=None, fit_win=1250, refit=20):
    """The design of Gibbs and Candes (2021) for stock volatility: V_t = r_t^2, sigma_t^2 from a GARCH(1,1) fitted on
    the previous fit_win days (re-estimated every `refit` days, filtered daily), score S_t = |V_t - sigma_t^2| /
    sigma_t^2. Returns dates, scores, sigma2, returns (from `start`)."""
    key = f'gs_{name}'
    if key in _MEM:
        return _MEM[key]
    start = start or GS_START
    r = log_returns(name)
    i0 = max(int(np.searchsorted(r.index, pd.Timestamp(start))), fit_win)
    rr = r.values
    sig2 = np.full(len(rr), np.nan)
    par = None
    for t in range(i0, len(rr)):
        if par is None or (t - i0) % refit == 0:
            par = garch11_fit(rr[t - fit_win:t], dist='normal')
        sig2[t] = garch11_filter(rr[t - fit_win:t], par)[-1]
    sl = slice(i0, len(rr))
    S = np.abs(rr[sl] ** 2 - sig2[sl]) / sig2[sl]
    _MEM[key] = (r.index[sl], S, sig2[sl], rr[sl])
    return _MEM[key]


def local_cov(miss, w=500):
    """Rolling coverage over the last w points (the local coverage frequency of Gibbs and Candes 2021)."""
    return 1 - pd.Series(miss).rolling(w).mean().values


def fig_aci(save_it=True, alpha=0.1, gamma=0.005, n_cal=1250):
    """S&P 500, BET and NVIDIA: local coverage (two-year window) of 90% volatility intervals: static split conformal
    (calibrated once on the first 1250 scores), rolling split conformal with a fixed level and ACI (gamma = 0.005)."""
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
    out = {}
    for ax, nm, lab in zip(axs, ('sp500', 'bet', 'nvda'), ('S&P 500', 'BET', 'NVIDIA')):
        dates, S, s2, r = garch_scores(nm)
        res = {}
        for meth, col, lb in (('static', st.Amber, 'static split'), ('rolling', st.MainBlue, 'rolling, fixed level'),
                              ('aci', st.IDAred, 'ACI')):
            q, at = online_threshold(S, alpha, method=meth, n_cal=n_cal, window=n_cal, gamma=gamma)
            miss = (S > q)[n_cal:].astype(float)
            lc = local_cov(miss)
            ax.plot(dates[n_cal:], lc, color=col, lw=1.2, label=lb)
            hits = miss
            res[meth] = dict(cov=float(1 - miss.mean()), lc_min=float(np.nanmin(lc)), lc_max=float(np.nanmax(lc)),
                             kup=kupiec(hits, alpha)[1], cc=christoffersen(hits, alpha)['p_cc'],
                             inf=float(np.mean(np.isinf(q[n_cal:]))))
        ax.axhline(1 - alpha, color=st.Forest, ls='--', lw=1)
        ax.set_title(lab)
        out[nm] = dict(res, n=int(len(S) - n_cal), first=str(dates[n_cal].date()), last=str(dates[-1].date()))
    axs[0].set_ylabel('local coverage (500 days)')
    st.fig_legend_bottom(fig, ncol=4, y=0.0)
    plt.tight_layout(rect=(0, 0.07, 1, 1))
    save('ats_ch13_aci', save_it)
    return out


def fig_aci_gamma(save_it=True, alpha=0.1, n_cal=1250):
    """ACI on the S&P 500 scores: the level alpha_t for gamma = 0.001, 0.005, 0.05; long-run miss rate, its bound
    (max(alpha_1, 1 - alpha_1) + gamma) / (gamma T), interval instability (s.d. of the threshold changes) and the
    share of infinite intervals."""
    dates, S, s2, r = garch_scores('sp500')
    fig, ax = plt.subplots(figsize=(11, 3.8))
    out = {}
    for g, col in ((0.001, st.MainBlue), (0.005, st.IDAred), (0.05, st.Forest)):
        q, at = online_threshold(S, alpha, method='aci', n_cal=n_cal, window=n_cal, gamma=g)
        miss = (S > q)[n_cal:]
        T = len(miss)
        ax.plot(dates[n_cal:], at[n_cal:], color=col, lw=1.0, label=f'gamma = {g}')
        qq = q[n_cal:][np.isfinite(q[n_cal:])]
        out[str(g)] = dict(miss=float(miss.mean()), bound=float((max(alpha, 1 - alpha) + g) / (g * T)),
                           inf=float(np.mean(np.isinf(q[n_cal:]))), amin=float(np.nanmin(at[n_cal:])),
                           amax=float(np.nanmax(at[n_cal:])), jitter=float(np.std(np.diff(qq))), T=T)
    ax.axhline(alpha, color=st.DarkText, ls=':', lw=1)
    ax.set_ylabel('ACI level alpha_t')
    st.legend_outside_bottom(ax, ncol=3, y=-0.14)
    plt.tight_layout()
    save('ats_ch13_aci_gamma', save_it)
    return out


def fig_condcov(save_it=True, alpha=0.1, n_cal=1250):
    """Conditional coverage on the S&P 500: coverage of the static, rolling and ACI intervals by tercile of the GARCH
    variance forecast and in the days after a miss (the dependence the Christoffersen test detects)."""
    dates, S, s2, r = garch_scores('sp500')
    terc = pd.qcut(s2[n_cal:], 3, labels=False)
    fig, ax = plt.subplots(figsize=(10, 3.8))
    out = {}
    labs = ['low variance', 'middle', 'high variance', 'day after a miss']
    for j, (meth, col, lb) in enumerate((('static', st.Amber, 'static split'), ('rolling', st.MainBlue, 'rolling'),
                                         ('aci', st.IDAred, 'ACI'))):
        q, _ = online_threshold(S, alpha, method=meth, n_cal=n_cal, window=n_cal, gamma=0.005)
        cov = (S <= q)[n_cal:]
        v = [float(cov[terc == k].mean()) for k in range(3)] + [float(cov[1:][~cov[:-1]].mean())]
        ax.bar(np.arange(4) + (j - 1) * 0.26, v, width=0.26, color=col, label=lb)
        out[meth] = v
    ax.axhline(1 - alpha, color=st.Forest, ls='--', lw=1.2, label='target 0.90')
    ax.set_xticks(range(4))
    ax.set_xticklabels(labs)
    ax.set_ylim(0.85, 0.94)
    ax.set_ylabel('coverage')
    st.legend_outside_bottom(ax, ncol=4, y=-0.14)
    plt.tight_layout()
    save('ats_ch13_condcov', save_it)
    return out


def load_baselines(win=LOAD_WIN, res_win=LOAD_RES, first=None):
    """Day-ahead forecasts for day d1 with data up to day d1 - 1: the weekly naive and the expert ARX of Ziel and
    Weron (2018) for each hour (as in Chapter 1); quantiles = point forecast plus empirical quantiles of the last
    res_win errors at the same hour. Returns days, actuals, points and decile quantiles from `first`."""
    first = first or LOAD_EVAL
    Y = ro_load_hourly()
    days, A = Y.index, Y.values
    nD = len(days)
    dow = np.array([d.dayofweek for d in days])
    start = int(np.searchsorted(days, pd.Timestamp(first)))
    P = {k: np.full((nD, 24), np.nan) for k in ('weekly naive', 'expert ARX')}
    for d1 in range(7, nD):
        P['weekly naive'][d1] = A[d1 - 7]
    mins, maxs, last = A.min(axis=1), A.max(axis=1), A[:, 23]
    for d1 in range(start - res_win - 1, nD):
        rows = np.arange(d1 - win, d1)
        for h in range(24):
            X = np.column_stack([np.ones(len(rows)), A[rows - 1, h], A[rows - 2, h], A[rows - 7, h], mins[rows - 1],
                                 maxs[rows - 1], last[rows - 1], dow[rows] == 0, dow[rows] == 5, dow[rows] == 6])
            b = np.linalg.lstsq(X, A[rows, h], rcond=None)[0]
            x = np.r_[1.0, A[d1 - 1, h], A[d1 - 2, h], A[d1 - 7, h], mins[d1 - 1], maxs[d1 - 1], last[d1 - 1],
                      dow[d1] == 0, dow[d1] == 5, dow[d1] == 6]
            P['expert ARX'][d1, h] = x @ b
    Q = {}
    for k in P:
        E = A - P[k]
        Q[k] = np.full((nD, 24, 9), np.nan)
        for d1 in range(start, nD):
            Q[k][d1] = P[k][d1][:, None] + np.nanquantile(E[d1 - res_win:d1], DECILES, axis=0).T
    return days[start:], A[start:], {k: v[start:] for k, v in P.items()}, {k: v[start:] for k, v in Q.items()}


def load_deep(first=None, L=336, refit_months=3, seed=SEED):
    """DLinear and N-BEATS (Chapter 12 architectures) trained on all hourly windows before each refit date (every
    refit_months months), inputs and targets divided by the mean of the input window; day-ahead point forecasts."""
    if not TORCH:
        return {}
    first = first or LOAD_EVAL
    Y = ro_load_hourly()
    days = Y.index
    y = Y.values.ravel()
    start = int(np.searchsorted(days, pd.Timestamp(first)))
    out = {k: np.full((len(days) - start, 24), np.nan) for k in ('DLinear', 'N-BEATS')}
    refits = pd.date_range(first, days[-1], freq=f'{refit_months}MS')
    for i, r in enumerate(refits):
        d0 = int(np.searchsorted(days, r))
        d1 = int(np.searchsorted(days, refits[i + 1])) if i + 1 < len(refits) else len(days)
        ends = np.arange(L, d0 * 24 - 24 + 1, 6)                                    # window ends (hours), stride 6
        X = np.stack([y[e - L:e] for e in ends])
        Yt = np.stack([y[e:e + 24] for e in ends])
        sc = X.mean(axis=1, keepdims=True)
        Xo = np.stack([y[d * 24 - L:d * 24] for d in range(d0, d1)])
        so = Xo.mean(axis=1, keepdims=True)
        for k, net in (('DLinear', DLinear(L, 24)), ('N-BEATS', NBeats(L, 24))):
            set_seed(seed)
            m = fit_global(net, X / sc, Yt / sc, epochs=30, seed=seed)
            out[k][d0 - start:d1 - start] = predict_global(m, Xo / so) * so
    return out


def load_fm(first=None, ctx=LOAD_CTX, names=FMS):
    """Zero-shot day-ahead decile forecasts of the foundation models: context = the last ctx hours up to the end of
    day d1 - 1."""
    first = first or LOAD_EVAL
    Y = ro_load_hourly()
    days = Y.index
    y = Y.values.ravel()
    start = int(np.searchsorted(days, pd.Timestamp(first)))
    C = [y[d * 24 - ctx:d * 24] for d in range(start, len(days))]
    out = {}
    for nm in names:
        Q = fm_cached(f'load_{first}_{ctx}', nm, C, 24, DECILES)
        if Q is not None:
            out[nm] = Q
    return out


def load_all():
    if 'load_all' in _MEM:
        return _MEM['load_all']
    days, A, P, Q = load_baselines()
    deep = load_deep()
    F = load_fm()
    for k, v in F.items():
        P[k] = v[:, :, 4]
        Q[k] = v
    P.update(deep)
    _MEM['load_all'] = (days, A, P, Q)
    return _MEM['load_all']


def load_scores(base='expert ARX'):
    """Hour-by-hour streams of absolute day-ahead errors of a base forecaster (days x 24) on Romanian load."""
    days, A, P, Q = load_all()
    return days, A, P[base], np.abs(A - P[base])


def fig_pid(save_it=True, alpha=0.1, n_cal=91, base='Chronos-Bolt small'):
    """90% day-ahead intervals for Romanian load around a base point forecast, one online threshold per hour of the
    day: static split (first n_cal days), rolling split (last n_cal days), ACI, quantile tracking, conformal PID
    (quantile tracking + integrator + AR(1) scorecaster) and EnbPI on the expert ARX regressors."""
    days, A, P, S = load_scores(base)
    nD = len(days)
    meths = [('static', 'static split'), ('rolling', 'rolling split'), ('aci', 'ACI'), ('qt', 'quantile tracking'),
             ('pid', 'conformal PID')]
    miss = {m: np.full((nD, 24), np.nan) for m, _ in meths}
    wid = {m: np.full((nD, 24), np.nan) for m, _ in meths}
    for h in range(24):
        s = S[:, h]
        for m, _ in meths:
            q, _ = online_threshold(s, alpha, method=m, n_cal=n_cal, window=n_cal, gamma=0.005,
                                    scorecaster=ar_scorecaster(1, 365) if m == 'pid' else None)
            miss[m][n_cal:, h] = (s > q)[n_cal:]
            wid[m][n_cal:, h] = 2 * q[n_cal:]
    # EnbPI on the expert ARX regressors, one stream per hour
    Y = ro_load_hourly()
    dd, AA = Y.index, Y.values
    dow = np.array([d.dayofweek for d in dd])
    st0 = int(np.searchsorted(dd, days[0])) - 365
    em, ew = np.full((nD, 24), np.nan), np.full((nD, 24), np.nan)
    mins, maxs, last = AA.min(1), AA.max(1), AA[:, 23]
    for h in range(24):
        rows = np.arange(st0, len(dd))
        X = np.column_stack([AA[rows - 1, h], AA[rows - 2, h], AA[rows - 7, h], mins[rows - 1], maxs[rows - 1],
                             last[rows - 1], dow[rows] == 0, dow[rows] == 5, dow[rows] == 6]).astype(float)
        c, hw = enbpi(X, AA[rows, h], 365, alpha, B=20, block=7, seed=SEED)
        em[:, h] = np.abs(AA[rows[365:], h] - c) > hw
        ew[:, h] = 2 * hw
    miss['enbpi'], wid['enbpi'] = em, ew
    meths.append(('enbpi', 'EnbPI (ridge ensemble)'))
    fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.0))
    out = {}
    cols = {'static': st.Amber, 'rolling': st.MainBlue, 'aci': st.IDAred, 'qt': st.Purple, 'pid': st.Forest, 'enbpi': st.Teal}
    for m, lb in meths:
        mm = np.nanmean(miss[m], axis=1)
        lc = 1 - pd.Series(mm).rolling(30).mean()
        axs[0].plot(days, lc, color=cols[m], lw=1.1, label=lb)
        out[m] = dict(cov=float(1 - np.nanmean(miss[m][n_cal:])), width=float(np.nanmean(wid[m][n_cal:])),
                      lc_min=float(np.nanmin(lc[n_cal:])))
        axs[1].scatter(out[m]['width'], out[m]['cov'], color=cols[m], s=70)
    axs[0].axhline(1 - alpha, color=st.DarkText, ls=':', lw=1)
    axs[0].set_ylabel('coverage, 30-day rolling')
    import matplotlib.dates as mdates
    axs[0].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 4, 7, 10)))
    axs[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    axs[0].tick_params(axis='x', labelsize=9.5)
    axs[1].axhline(1 - alpha, color=st.DarkText, ls=':', lw=1)
    axs[1].set_xlabel('mean width (GW)')
    axs[1].set_ylabel('coverage')
    st.fig_legend_bottom(fig, ncol=3, y=0.0)
    plt.tight_layout(rect=(0, 0.1, 1, 1))
    save('ats_ch13_pid', save_it)
    out['base'] = base
    out['first'] = str(days[n_cal].date())
    return out

## Run

In [ ]:
print(fig_weighted(reps=50))
print(fig_aci())
print(fig_aci_gamma())
print(fig_condcov())
print(fig_pid(n_cal=30))

![ats_ch13_weighted](./ats_ch13_weighted.png)

Output: `ats_ch13_weighted.pdf`

![ats_ch13_aci](./ats_ch13_aci.png)

Output: `ats_ch13_aci.pdf`

![ats_ch13_aci_gamma](./ats_ch13_aci_gamma.png)

Output: `ats_ch13_aci_gamma.pdf`

![ats_ch13_condcov](./ats_ch13_condcov.png)

Output: `ats_ch13_condcov.pdf`

![ats_ch13_pid](./ats_ch13_pid.png)

Output: `ats_ch13_pid.pdf`